In [1]:
#!module load cuda11/11.8
!nvidia-smi

NVIDIA-SMI has failed because it couldn't communicate with the NVIDIA driver. Make sure that the latest NVIDIA driver is installed and running.



In [2]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA version PyTorch built with:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))
else:
    print("CUDA not available")


import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt
import numpy as np
import os
print(os.environ.get('CONDA_DEFAULT_ENV'))

from tqdm import tqdm, trange

import sys
sys.path.append('../utilities')

import to_utils_3d
import models_torch_3d

PyTorch version: 2.5.1
CUDA version PyTorch built with: 12.1
CUDA available: False
CUDA not available
to_gan


In [5]:
data_path = "/home/u26/emcdugald/TO_Gan/TO_3D_data_scratch/data/labeled_voxels_32x32x32.npy"
# batch_size = 8
# nz = 100
# ngf = 32
# ndf = 32
# num_epochs = 1

P, N = to_utils_3d.load_data_3d(data_path)
n_samples = min(len(P), len(N))
P = P[:n_samples]
N = N[:n_samples]


# Optionally augment
# P = augment_all_3d(P)
# N = augment_all_3d(N)

Positives: 75 Negatives: 25


In [6]:
validity, _ = to_utils_3d.eval_batch_validity_3d(N.detach().numpy()[:,0])   # Remove channel for validity
assert np.all(validity == False)

validity, _ = to_utils_3d.eval_batch_validity_3d(P.detach().numpy()[:,0])
assert np.all(validity == True)

In [9]:
from models_torch_3d import GAN_step_MDD_3d
from torch.utils.data import TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

batch_size = 8
nz = 100
ngf = 32
ndf = 32
num_epochs = 1

shape3d = P.shape[2:]
netG = to_utils_3d.Generator3d(nz, ngf, shape3d).to(device)
netD = to_utils_3d.Discriminator3d(ndf, shape3d, nc=2).to(device)

P_loader = models_torch_3d.ReusableDataLoader(TensorDataset(P), batch_size)
N_loader = models_torch_3d.ReusableDataLoader(TensorDataset(N), batch_size)
num_steps = num_epochs * len(P) // batch_size

D_opt = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
G_opt = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Training loop
netD, netG, _ = models_torch_3d.train_3d(
    netD, netG, None, D_opt, G_opt, None, P_loader, N_loader, num_steps, batch_size, nz, GAN_step_MDD_3d, device, 1, 0
)

device: cpu


100%|██████████| 3/3 [00:02<00:00,  1.19it/s, L_D_real=0.8861, L_D_neg=0.9967, L_D_fake=1.0868, L_G=6.5763]


In [10]:

def plot_voxel_grid_3d(voxel_grid, title="", save_path=None):
    from mpl_toolkits.mplot3d import Axes3D
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.voxels(voxel_grid > 0, edgecolor='k', linewidth=0.2)
    ax.set_title(title)
    plt.axis('off')
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig)

# Generate and plot a batch of fake samples
netG.eval()
with torch.no_grad():
    fake = netG(torch.randn(batch_size, nz, device=device)).cpu().numpy()
    for idx in range(batch_size):
        plot_voxel_grid_3d(fake[idx, 0], title=f"Fake sample {idx}", save_path=f"fake_voxel_{idx}.png")
print(f"Saved voxel plots for {batch_size} fake samples.")

# Evaluate generator
all_valid, all_areas, all_div = to_utils_3d.evaluate_n_batches_3d(netG, device, nz, batches=10, batch_size=batch_size)
print("Mean invalidity rate (%):", (1-np.mean(all_valid))*100)
print("Mean violation magnitude (%):", np.mean(all_areas)/np.prod(shape3d)*100)
print("Mean diversity:", np.mean(all_div))